<h1 align="center">Práctica 4. Árboles binarios de búsqueda</h1>
<h2 align="center">Sesión 1: Clasificación de candidatos por nota</h2>
<h3 align="center">Estructuras de datos</h3>
<h3 align="center">Grado en Inteligencia Artificial</h3>
<h3 align="center">2026-2027 · Universitat Politècnica de València</h3>


## Índice
1. [Introducción](#intro)
2. [Código proporcionado y clase Candidato](#codigo)
3. [Actividad 1: Insertar y buscar candidatos](#act1)
4. [Actividad 2: Obtener la clasificación por nota](#act2)
5. [Actividad 3: Consultar un intervalo de notas](#act3)
6. [Comparación final: el orden de inserción](#final)


<a id="intro"></a>
## Introducción

En la práctica anterior utilizamos un montículo para conservar los mejores candidatos a un puesto de trabajo. Ahora vamos a guardar candidatos en un árbol binario de búsqueda (ABB o BST), obtener una clasificación por nota y consultar los candidatos de un intervalo.

Cada nodo guardará una **nota como clave** y el **candidato completo como valor**. Las notas son distintas: en esta práctica no hay empates.

En un ABB, todas las claves del subárbol izquierdo de un nodo son menores que su clave, y todas las del subárbol derecho son mayores o iguales. Para buscar una nota, comenzamos en la raíz: si coincide, hemos encontrado el candidato; si es menor, seguimos por la izquierda; si es mayor o igual, por la derecha.

Deja `candidatos2.csv` en la misma carpeta que este notebook. El fichero contiene 41 candidatos ficticios, sin cabecera, con los mismos campos de la práctica anterior: DNI, nombre completo, fecha de solicitud y nota. El separador es `;` y la codificación es UTF-8.


<a id="codigo"></a>
## Código proporcionado y clase `Candidato`

Ejecuta la celda de la clase `BST`. No tienes que modificarla. Para esta práctica utilizaremos:

- `b[nota] = candidato`: inserta un candidato asociado a una nota.
- `b[nota]`: consulta el candidato de esa nota; lanza `KeyError` si no existe.
- `nota in b`: comprueba si existe esa clave.
- `len(b)`: obtiene el número de nodos.
- `b.items()`: recorre los pares `(nota, candidato)` de menor a mayor nota.


In [15]:
class BST:
    class _Node:
        def __init__(self, key, value, left=None, right=None):
            self._key = key
            self._value = value
            self._left = left
            self._right = right
            
    def __init__(self): # create empty binary tree
        self._root = None
        self._size = 0

    def __len__(self):
        return self._size

    def __getitem__(self, key):
        node = self._root
        while node is not None:
            if key == node._key:
                return node._value
            elif key < node._key:
                node = node._left
            else:
                node = node._right
        raise KeyError("Key Error: " + repr(key))

    def __contains__(self, key):
        node = self._root
        while node is not None:
            if key == node._key:
                return True
            elif key < node._key:
                node = node._left
            else:
                node = node._right
        return False

    def __setitem__(self, key, value):
        prev, node = None, self._root
        while node is not None:
            prev = node
            if key == node._key:
                node._value = value
                return
            elif key < node._key:
                node = node._left
            else:
                node = node._right
        newchild = BST._Node(key, value)
        self._size += 1
        if prev is None:
            self._root = newchild
        elif key < prev._key:
            prev._left = newchild
        else:
            prev._right = newchild

    def find_gt_iter(self, key): # sucesor iterativo
        def aux(nodo):
            """
            es como un inorder pero únicamente visita claves > key
            """
            if nodo is not None:
                if nodo._key > key:
                    yield from aux(nodo._left)
                    yield (nodo._key, nodo._value)
                yield from aux(nodo._right)
        yield from aux(self._root)

    def find_gt(self, key): # sucesor
        return next(self.find_gt_iter(key), None)
    
    def __delitem__ (self, key):
        prev, node = None, self._root
        while node is not None and node._key != key:
            prev = node
            if key <  node._key:
                node = node._left
            else:
                node = node._right
        if node is None: # key not found
            raise KeyError("Key Error: " + repr(key))
        # key has been found, we have to remove it
        self._size -= 1
        if node._left is None or node._right is None:
             # one child of a leaf
            if node._left is not None:
                replace = node._left
            else:
                replace = node._right
            if prev is None:
                self._root = replace
            elif prev._left == node:
                prev._left = replace
            else:
                prev._right = replace
        else: # two children
            # el mínimo de su subárbol derecho ocupa su lugar:
            parentmin, themin = node, node._right
            while themin._left is not None:
                parentmin, themin = themin, themin._left
            node._key, node._value = themin._key, themin._value
            # remove themin:
            if parentmin._left == themin:
                parentmin._left = themin._right
            else:
                parentmin._right = themin._right

    def _inorder_traversal(self, node):
        if node is not None:
            yield from self._inorder_traversal(node._left)
            yield node
            yield from self._inorder_traversal(node._right)
                
    def __iter__(self): # recorre claves, lo que se ejecuta si iteras self
        for node in self._inorder_traversal(self._root):
            yield node._key
            
    def items(self): # recorre pares (tuplas) clave, valor
        for node in self._inorder_traversal(self._root):
            yield (node._key, node._value)

    # mostrar árbol como diccionario python
    def __repr__(self):
        return "{" + ",".join(repr(k)+':'+repr(v) for k,v in self.items()) + "}"

    
    # mostrar árbol como árbol mediante parentizado
    # ejemplo: '15(5(3(,),12(10(6(,7(,)),),13(,))),16(,20(18(,),23(,))))'
    def __aux_repr_tree__(self, node):
        if node is None:
            return ''
        return (repr(node._key) + '(' +
                self.__aux_repr_tree__(node._left)    + ',' +
                self.__aux_repr_tree__(node._right)   + ')')
        
    def __repr_tree__(self):
        return self.__aux_repr_tree__(self._root)
    
    def mostrar(self):
        def aux(nodo, prefijo, etiqueta):
            if nodo is not None:
                print(f'{prefijo}{etiqueta}: {nodo._key:g}')
                aux(nodo._left, prefijo + '    ', 'I')
                aux(nodo._right, prefijo + '    ', 'D')
        if self._root is None:
            print('(árbol vacío)')
        else:
            aux(self._root, '', 'Raíz')

**Pega en la siguiente celda la clase `Candidato` que implementaste en la práctica 3**, incluidas sus importaciones. Su constructor recibe DNI, nombre, fecha y nota, en ese orden.

En las celdas siguientes accedemos a la nota mediante `candidato.nota`. Si tu atributo tiene otro nombre, adapta ese acceso. La nota debe estar almacenada como `float`.


In [16]:
from datetime import datetime
from typing import Union
class Candidato:
    def __init__(self, dni: str, nombre: str, fecha: str, nota: Union[float, str]) -> None:
        self._dni: str = dni
        self._nombre: str = nombre
        self._fecha: datetime = datetime.strptime(fecha, '%Y-%m-%d %H:%M:%S')
        self.nota: float = float(nota)

    def __lt__(self, other) -> bool:
        if self.nota < other._nota:
            return True
        elif self.nota == other._nota:
            return self._fecha > other._fecha
        return False

    def __repr__(self):
        return  f"{self._dni},{self._nombre},{self._fecha.strftime('%Y-%m-%d %H:%M:%S')},{self.nota}"
            
    

El código de BST incluye un método `mostrar` que muestra las notas del árbol. `I` indica un hijo izquierdo y `D` un hijo derecho; la sangría indica el nivel.

<a id="act1"></a>
## Actividad 1: Insertar y buscar candidatos

### 1.1. Primeras inserciones

Ejecuta este ejemplo resuelto. Las notas 6 y 9 quedan a la izquierda y a la derecha de 7.5, respectivamente.


In [17]:
ejemplos = [
    Candidato('10000001A', 'Ana García López', '2026-03-01 09:00:00', '7.50'),
    Candidato('10000002B', 'Luis Pérez Ruiz', '2026-03-02 10:00:00', '6.00'),
    Candidato('10000003C', 'Marta Torres Gil', '2026-03-03 11:00:00', '9.00'),
    Candidato('10000004D', 'Pedro Romero Vidal', '2026-03-04 12:00:00', '5.50'),
    Candidato('10000005E', 'Elena Navarro Soler', '2026-03-05 13:00:00', '6.50'),
    Candidato('10000006F', 'Javier Gómez Molina', '2026-03-06 14:00:00', '8.00'),
    Candidato('10000007G', 'Lucía Fernández Cano', '2026-03-07 15:00:00', '9.50'),
]
pequeno = BST()
for candidato in ejemplos[:3]:
    pequeno[candidato.nota] = candidato
pequeno.mostrar()


Raíz: 7.5
    I: 6
    D: 9


Completa la inserción de los cuatro candidatos restantes y ejecuta la celda. El árbol debe tener siete nodos. La nota 5.5 quedará a la izquierda de 6; 6.5, a su derecha; 8, a la izquierda de 9; y 9.5, a su derecha.


In [18]:
for candidato in ejemplos[3:]:
    # insertar el candidato usando su nota como clave.
    pequeno[candidato.nota] = candidato

# Muestra el arbol
pequeno.mostrar()

#Comprueba las condiciones si no salta el error, entonces todo bien
assert len(pequeno) == 7
assert pequeno[6.5] is ejemplos[4]
assert 8.0 in pequeno
print('Inserciones correctas')


Raíz: 7.5
    I: 6
        I: 5.5
        D: 6.5
    D: 9
        I: 8
        D: 9.5
Inserciones correctas


### 1.2. Seguir una búsqueda

Completa el método `mostrar_busqueda`. El método imprime las notas visitadas y devuelve el candidato si lo encuentra. Si la nota no existe, imprime el mensaje indicado y devuelve `None`.

El bucle y los mensajes están preparados. Añade los tres casos: coincidencia, continuar por la izquierda y continuar por la derecha. Puedes basarte en `__getitem__` de la clase `BST`.


In [19]:
class BST_Seguir(BST): # clase que hereda de BST
    def mostrar_busqueda(self, nota):
        nodo = self._root
        while nodo is not None:
            print(f'Visitamos la nota {nodo._key:g}')
            if nota == nodo._key:
                return nodo._value
            elif nodo._key < nota:
                nodo = nodo._right
            else:
                nodo = nodo._left
            
        print('No existe un candidato con esa nota')
        return None

Para buscar 8, deben visitarse las notas 7.5, 9 y 8. Para buscar 6.8, se visitan 7.5, 6 y 6.5, y después se indica que no existe. Ejecuta las consultas:


In [20]:
# tenemos que reconvertir pequeno en la nueva clase:
pequeno = BST_Seguir()
for candidato in ejemplos:
    pequeno[candidato.nota] = candidato
    
vacio = BST_Seguir() # un árbol vacío

print('Búsqueda de 8:')
encontrado = pequeno.mostrar_busqueda(8.0)
assert encontrado is ejemplos[5]
print('Candidato:', encontrado)

print('\nBúsqueda de 6.8:')
assert pequeno.mostrar_busqueda(6.8) is None
assert pequeno.mostrar_busqueda(7.5) is ejemplos[0]
assert vacio.mostrar_busqueda(7.5) is None
print('Búsquedas correctas')


Búsqueda de 8:
Visitamos la nota 7.5
Visitamos la nota 9
Visitamos la nota 8
Candidato: 10000006F,Javier Gómez Molina,2026-03-06 14:00:00,8.0

Búsqueda de 6.8:
Visitamos la nota 7.5
Visitamos la nota 6
Visitamos la nota 6.5
No existe un candidato con esa nota
Visitamos la nota 7.5
No existe un candidato con esa nota
Búsquedas correctas


### 1.3. Cargar el fichero de candidatos

El siguiente código lee `candidatos2.csv` y crea los objetos `Candidato`. Ejecuta la celda:


In [21]:
import csv

candidatos = []
with open('candidatos2.csv', encoding='utf-8', newline='') as f:
    for fila in csv.reader(f, delimiter=';'):
        candidatos.append(Candidato(*fila))

assert len(candidatos) == 41
assert len({c.nota for c in candidatos}) == len(candidatos)
print('Candidatos leídos:', len(candidatos))


Candidatos leídos: 41


Completa la carga de los candidatos en un nuevo árbol de la clase `BST_Sequir`, respetando el orden del fichero. Debe haber un nodo por candidato.


In [23]:
arbol = BST_Seguir()
for candidato in candidatos:
    arbol[candidato.nota] = candidato
assert len(arbol) == len(candidatos)
assert arbol[5.0].nota == 5.0
assert arbol[9.8].nota == 9.8
print('Candidatos en el árbol:', len(arbol))


Candidatos en el árbol: 41


<a id="act2"></a>
## Actividad 2: Obtener la clasificación por nota

El recorrido **inorden** visita primero el subárbol izquierdo, después el nodo y finalmente el subárbol derecho. En un ABB permite obtener las claves de menor a mayor.

El método `items()` ya proporciona ese recorrido. Completa `candidatos_ordenados(arbol)` para devolver una **lista de candidatos**, de menor a mayor nota. Un árbol vacío debe dar `[]`.

Recorre los pares `(nota, candidato)` y añade sus candidatos al resultado. No utilices `sorted` ni `sort`: el árbol ya proporciona el orden.

En el árbol pequeño, las notas del resultado serán `[5.5, 6.0, 6.5, 7.5, 8.0, 9.0, 9.5]`.


In [24]:
def candidatos_ordenados(arbol):
    resultado = []
    # recorrer arbol.items() y añadir cada candidato al resultado.
    for nota, candidato in arbol.items():
        resultado.append(candidato)

    return resultado

In [25]:
ordenados_pequeno = candidatos_ordenados(pequeno)
assert [c.nota for c in ordenados_pequeno] == [5.5, 6.0, 6.5, 7.5, 8.0, 9.0, 9.5]
assert ordenados_pequeno[0] is ejemplos[3]
assert candidatos_ordenados(BST_Seguir()) == []

clasificacion = candidatos_ordenados(arbol)
assert len(clasificacion) == 41
assert [c.nota for c in clasificacion] == [round(5.0 + i * 0.12, 2) for i in range(41)]
assert {id(c) for c in clasificacion} == {id(c) for c in candidatos}
print('Clasificación correcta')


Clasificación correcta


Para mostrar primero a los candidatos con mayor nota, podemos recorrer la lista anterior al revés con `reversed`. Ejecuta esta celda, que muestra los cinco primeros de la clasificación descendente:


In [26]:
mejores_primero = list(reversed(clasificacion))
for candidato in mejores_primero[:5]:
    print(candidato)
assert [c.nota for c in mejores_primero[:5]] == [9.8, 9.68, 9.56, 9.44, 9.32]


41005480E,Carlos Ruiz Sánchez,2026-03-24 00:20:00,9.8
41005343T,Alba Fernández García,2026-03-06 23:57:00,9.68
41005206R,Raúl Gómez Navarro,2026-02-17 23:34:00,9.56
41005069W,Irene Pérez Gómez,2026-05-01 23:11:00,9.44
41004932A,Hugo Sánchez López,2026-04-14 22:48:00,9.32


<a id="act3"></a>
## Actividad 3: Consultar un intervalo de notas

Completa el método `candidatos_entre(minimo, maximo)`. Devuelve una lista de candidatos cuyas notas estén entre ambos límites, **incluidos**, de menor a mayor nota.

- Los límites no tienen por qué existir como claves del árbol.
- Si no hay resultados, el árbol está vacío o `minimo > maximo`, devuelve `[]`.
Implementa directamente un **inorden adaptado al intervalo**, completando la función auxiliar recursiva `aux(nodo)`. Comienza en `arbol._root` y trabaja con `nodo._key`, `nodo._value`, `nodo._left` y `nodo._right`.

Mantén el orden izquierda, nodo, derecha, pero **descarta los subárboles que no pueden contener notas del intervalo**:

- Si la nota del nodo es menor o igual que `minimo`, todas las notas de su subárbol izquierdo quedan fuera del intervalo. No lo visites.
- Añade el candidato del nodo solo si su nota está entre los límites, incluidos.
- Si la nota del nodo es mayor o igual que `maximo`, todas las notas de su subárbol derecho quedan fuera del intervalo. No lo visites.

**No utilices `items()`, la iteración del árbol ni `_inorder_traversal`.** Tampoco sirve visitar siempre todos los nodos y filtrar después. El caso de subárbol vacío ya está resuelto en el esqueleto. Las llamadas recursivas al auxiliar tienen la forma `aux(nodo._left)` o `aux(nodo._right)`; decide cuándo hacer cada una.

Por ejemplo, en el árbol pequeño, el intervalo `[6, 8]` contiene las notas `[6.0, 6.5, 7.5, 8.0]`. El recorrido no debe entrar en el nodo 5.5 ni en el nodo 9.5.


In [27]:
class BST_Entre(BST_Seguir): # clase que hereda de BST_Seguir

    def candidatos_entre(self, minimo, maximo):
        resultado = []
        if minimo > maximo:
            return resultado
        def aux(nodo):
            if nodo is None:
                return
            
            # 1. Visitar la izquierda si puede haber notas >= minimo
            if nodo._key > minimo:
                aux(nodo._left)

            # 2. Añadir el candidato si está dentro del intervalo
            if minimo <= nodo._key <= maximo:
                resultado.append(nodo._value)

            # 3. Visitar la derecha si puede haber notas <= maximo
            if nodo._key < maximo:
                aux(nodo._right)

        aux(self._root)
        return resultado


In [28]:
# tenemos que reconvertir pequeno en la nueva clase:
pequeno = BST_Entre()
for candidato in ejemplos:
    pequeno[candidato.nota] = candidato
    
vacio = BST_Entre() # un árbol vacío

casos = [
    (6.0, 8.0, [6.0, 6.5, 7.5, 8.0]),
    (6.1, 7.6, [6.5, 7.5]),
    (7.5, 7.5, [7.5]),
    (7.0, 7.0, []),
    (9.6, 10.0, []),
    (0.0, 5.4, []),
    (8.0, 6.0, []),
    (0.0, 10.0, [5.5, 6.0, 6.5, 7.5, 8.0, 9.0, 9.5]),
]

for minimo, maximo, esperado in casos:
    resultado = pequeno.candidatos_entre(minimo, maximo)
    assert [c.nota for c in resultado] == esperado, (minimo, maximo)
assert vacio.candidatos_entre(6.0, 8.0) == []
print('Consultas por intervalo correctas')


Consultas por intervalo correctas


Consulta ahora los candidatos del fichero con nota entre 8 y 9. Deben aparecer nueve candidatos, con notas desde 8.00 hasta 8.96:


In [29]:
arbol = BST_Entre() # recalculamos arbol para que utilice la nueva clase
for candidato in candidatos:
    arbol[candidato.nota] = candidato

seleccion = arbol.candidatos_entre(8.0, 9.0)
for candidato in seleccion:
    print(candidato)
assert [c.nota for c in seleccion] == [8.0, 8.12, 8.24, 8.36, 8.48, 8.6, 8.72, 8.84, 8.96]
print('Número de candidatos seleccionados:', len(seleccion))


41003425Z,Elena Sánchez López,2026-04-07 18:35:00,8.0
41003562J,Javier Pérez Gómez,2026-04-24 18:58:00,8.12
41003699N,Lucía Gómez Navarro,2026-02-10 19:21:00,8.24
41003836B,Sergio Fernández García,2026-02-27 19:44:00,8.36
41003973X,Paula Ruiz Sánchez,2026-03-16 20:07:00,8.48
41004110D,Diego Navarro Fernández,2026-04-02 20:30:00,8.6
41004247P,Clara Torres Torres,2026-04-19 20:53:00,8.72
41004384F,Mario Romero Martínez,2026-02-05 21:16:00,8.84
41004521Y,Nuria García Pérez,2026-02-22 21:39:00,8.96
Número de candidatos seleccionados: 9


<a id="final"></a>
## Comparación final: el orden de inserción

Ejecuta esta última celda. Construye otro árbol con los mismos siete candidatos del ejemplo, pero insertados por nota creciente, y muestra la búsqueda de la nota 9.5 en ambos.

La clasificación será la misma, aunque las formas de los árboles sean distintas. En el primero se visitan tres nodos para encontrar 9.5; en el segundo se visitan siete.


In [30]:
creciente = BST_Entre()
for candidato in candidatos_ordenados(pequeno):
    creciente[candidato.nota] = candidato

print('Árbol insertado por nota creciente:')
creciente.mostrar()
print('\nBúsqueda en el árbol inicial:')
assert pequeno.mostrar_busqueda(9.5) is ejemplos[6]
print('\nBúsqueda en el árbol insertado por nota creciente:')
assert creciente.mostrar_busqueda(9.5) is ejemplos[6]
assert candidatos_ordenados(creciente) == candidatos_ordenados(pequeno)


Árbol insertado por nota creciente:
Raíz: 5.5
    D: 6
        D: 6.5
            D: 7.5
                D: 8
                    D: 9
                        D: 9.5

Búsqueda en el árbol inicial:
Visitamos la nota 7.5
Visitamos la nota 9
Visitamos la nota 9.5

Búsqueda en el árbol insertado por nota creciente:
Visitamos la nota 5.5
Visitamos la nota 6
Visitamos la nota 6.5
Visitamos la nota 7.5
Visitamos la nota 8
Visitamos la nota 9
Visitamos la nota 9.5


**Antes de entregar:** completa las celdas indicadas y ejecuta todo desde un kernel reiniciado. Las comprobaciones deben terminar sin errores.
